# 🔍 Interactive Deepfake Image Predictor & Model Tester

This notebook allows you to test and compare different ViT / ConvNeXt models on any single face image:
- **Fine-tuned ViT Classifier** (`best_model_v3.pt` - 28.69M params, fine-tuned on 129k train set)
- **Pretrained ViT Backbone** (`model-3.safetensors` / `weights/model.safetensors` - 28.7M params DINOv3 SwiGLU)
- **ConvNeXt-Tiny Classifier** (`convnext_weakfix_v3.pt` / `best_convnext_tiny_trained.pt`)

You can easily switch models and test any image (e.g. `data/test/Untitled.jpg` or your custom image).

### 1. Environment Initialization & Checkpoint Discovery

In [ ]:
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import importlib.util

# Dynamic project root resolution
ROOT_DIR = Path(os.getcwd()).resolve()
if not (ROOT_DIR / 'src').exists() and (ROOT_DIR.parent / 'src').exists():
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Execution Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  • GPU: {torch.cuda.get_device_name(0)}')

# List available checkpoint files
ckpt_dir = ROOT_DIR / 'experiments/checkpoints'
available_ckpts = sorted([p for p in ckpt_dir.glob('*') if p.suffix in ['.pt', '.safetensors']])
print(f'\n📁 Available Checkpoints in {ckpt_dir.name}/:')
for p in available_ckpts:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'  • {p.name:<32} ({size_mb:6.2f} MB)')


### 2. Model Loader (Supports ViT .pt, .safetensors, & ConvNeXt)

In [ ]:
def load_deepfake_model(model_filename='best_model_v3.pt', device=DEVICE):
    """
    Loads a deepfake classification model from experiments/checkpoints/.
    Supports:
      - 'best_model_v3.pt' (Fine-tuned DINOv3 ViT-Small/16 classifier)
      - 'model-3.safetensors' (Pre-trained DINOv3 ViT backbone + classifier head)
      - 'convnext_weakfix_v3.pt' / 'best_convnext_tiny_trained.pt' (ConvNeXt-Tiny classifier)
    """
    model_path = ROOT_DIR / 'experiments/checkpoints' / model_filename
    if not model_path.exists():
        raise FileNotFoundError(f'Model checkpoint not found at: {model_path}')
    
    # 1. Load ViT .safetensors
    if model_path.suffix == '.safetensors':
        spec = importlib.util.spec_from_file_location('dinov3_vit', ROOT_DIR / 'src/models/dinov3_vit.py')
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        model = mod.build_dinov3_classifier(weights_path=str(model_path), num_classes=2, img_size=256, device=device)
        model.eval()
        print(f'✅ Loaded DINOv3 ViT Backbone from {model_path.name}')
        print(f'  • Architecture: DINOv3 ViT-Small (Gated MLP / SwiGLU)')
        print(f'  • Total Params: {sum(p.numel() for p in model.parameters()):,}')
        return model
    
    # 2. Load .pt checkpoint
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    
    # Determine if ViT or ConvNeXt
    is_convnext = any('downsample_layers' in k or 'stages' in k for k in state_dict.keys())
    
    if is_convnext:
        spec_cnn = importlib.util.spec_from_file_location('dinov3_convnext', ROOT_DIR / 'src/models/dinov3_convnext.py')
        mod_cnn = importlib.util.module_from_spec(spec_cnn)
        spec_cnn.loader.exec_module(mod_cnn)
        spec_cls = importlib.util.spec_from_file_location('classifier_v2', ROOT_DIR / 'src/models/classifier_v2.py')
        mod_cls = importlib.util.module_from_spec(spec_cls)
        spec_cls.loader.exec_module(mod_cls)
        
        backbone = mod_cnn.DinoConvNext()
        model = mod_cls.DinoConvNextClassifier(backbone, num_classes=2, hidden_dim=384)
        model.load_state_dict(state_dict, strict=False)
        model = model.to(device).eval()
        print(f'✅ Loaded DINOv3 ConvNeXt-Tiny from {model_path.name}')
        print(f'  • Total Params: {sum(p.numel() for p in model.parameters()):,}')
    else:
        spec = importlib.util.spec_from_file_location('dinov3_vit', ROOT_DIR / 'src/models/dinov3_vit.py')
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        
        model = mod.build_dinov3_classifier(weights_path=None, num_classes=2, img_size=256, device=device)
        model.load_state_dict(state_dict, strict=False)
        model = model.to(device).eval()
        print(f'✅ Loaded Fine-Tuned DINOv3 ViT-Small/16 from {model_path.name}')
        print(f'  • Total Params: {sum(p.numel() for p in model.parameters()):,}')
        if isinstance(ckpt, dict) and 'best_val_auc' in ckpt:
            val_auc = ckpt['best_val_auc']
            print(f'  • Best Val AUC: {val_auc * 100:.2f}%')
    
    return model

# Select and load your model (Change filename as desired):
MODEL_FILENAME = 'best_model_v3.pt'  # Options: 'best_model_v3.pt', 'model-3.safetensors', 'convnext_weakfix_v3.pt'
model = load_deepfake_model(MODEL_FILENAME)


### 3. Image Preprocessing & Dual-Panel Visualizer

In [ ]:
# Image preprocessing pipeline (256x256 Bicubic + ImageNet Norm)
transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_single_image(model, img_path, device=DEVICE):
    img_path = Path(img_path)
    if not img_path.exists():
        raise FileNotFoundError(f'Image not found at: {img_path}')
    
    raw_img = Image.open(img_path).convert('RGB')
    tensor_img = transform(raw_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(tensor_img)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()[0]
    
    p_real = probs[0]
    p_fake = probs[1]
    pred_label = 'FAKE (Deepfake)' if p_fake >= 0.5 else 'REAL (Authentic)'
    confidence = max(p_real, p_fake) * 100
    theme_color = '#c0392b' if p_fake >= 0.5 else '#27ae60'
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={'width_ratios': [1, 1.2]})
    
    ax1.imshow(raw_img)
    ax1.set_title(f'Input: {img_path.name}', fontsize=11, fontweight='bold')
    ax1.axis('off')
    
    bars = ax2.barh(['Real (Authentic)', 'Fake (Deepfake)'], [p_real * 100, p_fake * 100], color=['#27ae60', '#c0392b'], height=0.45, edgecolor='black', alpha=0.88)
    ax2.set_xlim(0, 105)
    ax2.set_xlabel('Probability (%)', fontsize=11, fontweight='bold')
    ax2.set_title(f'Prediction: {pred_label}\nConfidence: {confidence:.2f}%', fontsize=12, fontweight='bold', color=theme_color, pad=12)
    
    for bar in bars:
        w = bar.get_width()
        ax2.text(w + 2, bar.get_y() + bar.get_height() / 2, f'{w:.2f}%', va='center', ha='left', fontsize=10, fontweight='bold')
    
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return {
        'image': img_path.name,
        'prediction': pred_label,
        'p_real': float(p_real),
        'p_fake': float(p_fake),
        'confidence_percent': float(confidence)
    }


### 4. Interactive Test on Single Image
Specify any image path below and run the cell to test:

In [ ]:
# Specify the image to test:
TEST_IMAGE_PATH = ROOT_DIR / 'data/test/Untitled.jpg'

print(f'🔍 Testing image: {TEST_IMAGE_PATH}')
if TEST_IMAGE_PATH.exists():
    res = predict_single_image(model, TEST_IMAGE_PATH)
    print(f'📊 Result: {res["prediction"]} (Confidence: {res["confidence_percent"]:.2f}%)')
    print(f'   Real: {res["p_real"]*100:.2f}% | Fake: {res["p_fake"]*100:.2f}%')
else:
    print(f'⚠️ File not found at: {TEST_IMAGE_PATH}')
